In [3]:
import pandas as pd
import numpy as np

# Correct path — no ../ needed
df = pd.read_csv('data/processed/texas_part_d_2023.csv', low_memory=False)

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Shape: (1792881, 22)
Columns: ['Prscrbr_NPI', 'Prscrbr_Last_Org_Name', 'Prscrbr_First_Name', 'Prscrbr_City', 'Prscrbr_State_Abrvtn', 'Prscrbr_State_FIPS', 'Prscrbr_Type', 'Prscrbr_Type_Src', 'Brnd_Name', 'Gnrc_Name', 'Tot_Clms', 'Tot_30day_Fills', 'Tot_Day_Suply', 'Tot_Drug_Cst', 'Tot_Benes', 'GE65_Sprsn_Flag', 'GE65_Tot_Clms', 'GE65_Tot_30day_Fills', 'GE65_Tot_Drug_Cst', 'GE65_Tot_Day_Suply', 'GE65_Bene_Sprsn_Flag', 'GE65_Tot_Benes']


In [4]:
print(f"Rows before cleaning: {len(df)}")

# Drop rows where Tot_Benes is null
df = df[df['Tot_Benes'].notna()]

# Drop rows where Tot_Drug_Cst is null or zero
df = df[df['Tot_Drug_Cst'] > 0]

# Drop rows where Tot_Clms is null or zero
df = df[df['Tot_Clms'] > 0]

print(f"Rows after cleaning: {len(df)}")
print(f"Rows dropped: {1792881 - len(df)}")

Rows before cleaning: 1792881
Rows after cleaning: 869717
Rows dropped: 923164


In [6]:
# Fill suppression flag nulls with 'N' — null means not suppressed
df['GE65_Sprsn_Flag'] = df['GE65_Sprsn_Flag'].fillna('N')
df['GE65_Bene_Sprsn_Flag'] = df['GE65_Bene_Sprsn_Flag'].fillna('N')

# Fill GE65 numeric nulls with 0
ge65_cols = ['GE65_Tot_Clms', 'GE65_Tot_30day_Fills', 
             'GE65_Tot_Drug_Cst', 'GE65_Tot_Day_Suply', 'GE65_Tot_Benes']

df[ge65_cols] = df[ge65_cols].fillna(0)

print("Suppression flags handled")
print(f"\nGE65_Sprsn_Flag value counts:")
print(df['GE65_Sprsn_Flag'].value_counts())
print(f"\nGE65_Bene_Sprsn_Flag value counts:")
print(df['GE65_Bene_Sprsn_Flag'].value_counts())

Suppression flags handled

GE65_Sprsn_Flag value counts:
GE65_Sprsn_Flag
#    432414
N    407543
*     29760
Name: count, dtype: int64

GE65_Bene_Sprsn_Flag value counts:
GE65_Bene_Sprsn_Flag
#    525657
N    220500
*    123560
Name: count, dtype: int64


In [7]:
# Standardize text columns
text_cols = ['Prscrbr_Type', 'Gnrc_Name', 'Brnd_Name', 
             'Prscrbr_City', 'Prscrbr_Last_Org_Name']

for col in text_cols:
    df[col] = df[col].str.strip().str.title()

print("Text columns standardized ✓")
print(f"\nTop 10 specialties:")
print(df['Prscrbr_Type'].value_counts().head(10))

Text columns standardized ✓

Top 10 specialties:
Prscrbr_Type
Family Practice        229650
Internal Medicine      165736
Nurse Practitioner     164158
Physician Assistant     65827
Cardiology              29494
Emergency Medicine      17638
Dentist                 15300
Ophthalmology           11702
Endocrinology           10338
Nephrology              10215
Name: count, dtype: int64


In [9]:
# Cost per claim — high values relative to peers signal upcoding
df['cost_per_claim'] = df['Tot_Drug_Cst'] / df['Tot_Clms']

# Days supply per claim — abnormally high = possible fraud
df['days_per_claim'] = df['Tot_Day_Suply'] / df['Tot_Clms']

# Brand flag — 1 if brand name drug was prescribed, 0 if generic
df['is_brand'] = (
    df['Brnd_Name'].notna() & 
    (df['Brnd_Name'].str.strip() != '') &
    (df['Brnd_Name'].str.lower() != df['Gnrc_Name'].str.lower())
).astype(int)

print("Derived columns added")
print(f"\nNew column stats:")
print(df[['cost_per_claim', 'days_per_claim', 'is_brand']].describe())
print(f"\nBrand vs Generic distribution:")
print(df['is_brand'].value_counts())

Derived columns added

New column stats:
       cost_per_claim  days_per_claim       is_brand
count   869717.000000   869717.000000  869717.000000
mean       101.132947       48.526074       0.238199
std        501.366921       29.675391       0.425982
min          0.147368        0.913043       0.000000
25%          8.656667       22.181818       0.000000
50%         14.338500       50.266094       0.000000
75%         25.183958       77.140033       0.000000
max      81675.118000      365.000000       1.000000

Brand vs Generic distribution:
is_brand
0    662551
1    207166
Name: count, dtype: int64


In [10]:
# Flag statistically impossible or extreme day supply values
impossible_supply = df['days_per_claim'] > 180
print(f"Rows with >180 days per claim: {impossible_supply.sum()}")

# Flag extreme cost per claim outliers
extreme_cost = df['cost_per_claim'] > 10000
print(f"Rows with >$10,000 cost per claim: {extreme_cost.sum()}")

# Show sample of most suspicious rows
print(f"\nTop 10 most expensive cost-per-claim rows:")
df[extreme_cost][['Prscrbr_NPI', 'Prscrbr_Type', 
                   'Gnrc_Name', 'Tot_Clms',
                   'Tot_Drug_Cst', 'cost_per_claim']].sort_values(
                   'cost_per_claim', ascending=False).head(10)

Rows with >180 days per claim: 39
Rows with >$10,000 cost per claim: 332

Top 10 most expensive cost-per-claim rows:


,Prscrbr_NPI,Prscrbr_Type,Gnrc_Name,Tot_Clms,Tot_Drug_Cst,cost_per_claim
499326,1285016899,Neurology,Cladribine,25,2041877.95,81675.118000
1036871,1578873857,Rheumatology,Ravulizumab-Cwvz,76,5726116.93,75343.643816
1561282,1871547596,Rheumatology,Pegloticase,224,12690483.55,56653.944420
865293,1487636205,Endocrinology,Mifepristone,110,4543364.67,41303.315182
136427,1073959326,Endocrinology,Mifepristone,56,2007698.19,35851.753393
64394,1033640685,Physician Assistant,Mifepristone,64,2223972.25,34749.566406
433954,1245390210,Gastroenterology,Ustekinumab,131,4452589.44,33989.232366
444192,1255305256,Optometry,Cenegermin-Bkbj,48,1524097.70,31752.035417
1252960,1699872119,Gastroenterology,Sofosbuvir/Velpatasvir,29,841059.00,29002.034483
1482148,1821746835,Nurse Practitioner,Cabozantinib S-Malate,75,2123719.45,28316.259333


In [11]:
df.to_csv('data/processed/texas_part_d_clean.csv', index=False)
print(f"Clean dataset saved")
print(f"Final shape: {df.shape}")
print(f"New columns added: cost_per_claim, days_per_claim, is_brand")

Clean dataset saved
Final shape: (869717, 25)
New columns added: cost_per_claim, days_per_claim, is_brand
